In [2]:
import os
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [30]:
DATASET_PATH = r"C:\PrivDiffuser\datasets\DatasetIMUandBIOMARKERS"

print("Dataset exists:", os.path.exists(DATASET_PATH))
print()

print("Files/Folders inside DatasetIMUandBIOMARKERS:")
print("-" * 60)

for item in os.listdir(DATASET_PATH):
    print(item)

Dataset exists: True

Files/Folders inside DatasetIMUandBIOMARKERS:
------------------------------------------------------------
Subject01
Subject02
Subject03
Subject04
Subject05
Subject06
Subject07
Subject08
Subject09
Subject10
Subject11
Subject12
Subject13
Subject14
Subject15
Subject16
Subject17
Subject18
Subject19
Subject20
Subject21
Subject22
Subject23
Subject24
Subject25
Subject26
Subject27
Subject28
Subject29
Subject30
Subject31
Subject32
Subject33
Subject34
Subject35
Subject36
Subject37
Subject38
Subject39
Subject40
Subject41
Subject42
Subject43
Subject44
Subject45
Subject46
Subject47
Subject48
Subject49
Subject50
Subject51
Subject52
Subject53
Subject54
Subject55
Subject56
Subject57
Subject58
Subject59
Subject60
Subject61
Subject62
Subject63
Subject64
Subject65
Subject66
Subject67
SubjectsInfo.xlsx


In [58]:
windows.nbytes / (1024**2)

316.69921875

In [31]:
# ==========================================================
# Cell 2 - Configuration for Local and Narval Environments
# ==========================================================

import os
from pathlib import Path

# Local default:
# ../datasets/DatasetIMUandBIOMARKERS
#
# Narval:
# Set IMU_DATASET_PATH before launching Jupyter or the job.

DATASET_PATH = Path(
    os.environ.get(
        "IMU_DATASET_PATH",
        "../datasets/DatasetIMUandBIOMARKERS"
    )
).expanduser().resolve()

METADATA_FILE = DATASET_PATH / "SubjectsInfo.xlsx"

WINDOW_SIZE = 128
STRIDE = 10
RANDOM_STATE = 42

print("=" * 65)
print("CONFIGURATION")
print("=" * 65)

print("Dataset path :", DATASET_PATH)
print("Metadata file:", METADATA_FILE)
print("Window size  :", WINDOW_SIZE)
print("Stride       :", STRIDE)
print("Random state :", RANDOM_STATE)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset directory was not found:\n{DATASET_PATH}\n\n"
        "Set the IMU_DATASET_PATH environment variable "
        "to the dataset location."
    )

if not METADATA_FILE.exists():
    raise FileNotFoundError(
        f"Metadata file was not found:\n{METADATA_FILE}"
    )

print("\nDataset and metadata file verified successfully.")

Dataset Path : C:\PrivDiffuser\datasets\DatasetIMUandBIOMARKERS
Metadata File: C:\PrivDiffuser\datasets\DatasetIMUandBIOMARKERS\SubjectsInfo.xlsx
Window Size  : 128
Stride       : 10


In [32]:
# Cell 3 : Load Metadata

metadata = pd.read_excel(METADATA_FILE)
print("Metadata Loaded Successfully")

print("Number of Subjects :", len(metadata))

display(metadata.head())

Metadata Loaded Successfully
Number of Subjects : 60


,Participant ID,Gender,Age,Height (m),Weight (kg),Fat %,BMI,SpO2_baseline(%),HR_baseline(bpm),HR step test(bpm)
0,1,M,24,1.75,73.2,0.170,23.902041,96,98,128
1,2,M,27,1.80,70.2,0.132,21.666667,98,60,108
2,3,F,24,1.78,62.2,0.292,19.631360,98,89,132
3,4,F,23,1.65,60.6,0.273,22.258953,91,111,156
4,5,F,58,1.69,61.8,0.300,21.637898,98,67,88


In [33]:
# ==========================================================
# Cell 4 : Create Subject Folder Names
# ==========================================================

metadata["Subject"] = metadata["Participant ID"].apply(
    lambda x: f"Subject{x:02d}"
)

print("=" * 60)
print("Subject IDs Created")
print("=" * 60)

display(metadata[["Participant ID", "Subject"]].head())

print("\nTotal Subjects :", len(metadata))

Subject IDs Created


,Participant ID,Subject
0,1,Subject01
1,2,Subject02
2,3,Subject03
3,4,Subject04
4,5,Subject05



Total Subjects : 60


In [34]:
# ==========================================================
# Cell 5 : Train/Test Split
# ==========================================================

from sklearn.model_selection import train_test_split

train_subjects, test_subjects = train_test_split(
    metadata["Subject"].tolist(),
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("=" * 60)
print("Train/Test Split")
print("=" * 60)

print("Training Subjects :", len(train_subjects))
print("Testing Subjects  :", len(test_subjects))

print("\nFirst 5 Training Subjects")
print(train_subjects[:5])

print("\nFirst 5 Testing Subjects")
print(test_subjects[:5])

Train/Test Split
Training Subjects : 48
Testing Subjects  : 12

First 5 Training Subjects
['Subject36', 'Subject04', 'Subject58', 'Subject18', 'Subject09']

First 5 Testing Subjects
['Subject01', 'Subject06', 'Subject41', 'Subject51', 'Subject14']


In [37]:
# ==========================================================
# Cell 6 : Load IMU Data for One Subject
# ==========================================================

def load_subject_imu(subject):
    """
    Load one subject's IMU data.

    Parameters
    ----------
    subject : str
        Example: 'Subject01'

    Returns
    -------
    DataFrame
        IMU sensor dataframe
    """

    imu_path = os.path.join(
        DATASET_PATH,
        subject,
        f"IMU{subject}.csv"
    )

    df = pd.read_csv(imu_path)

    return df

In [38]:
# ==========================================================
# Cell 7 : Verify IMU Loading
# ==========================================================

subject = train_subjects[0]

imu_df = load_subject_imu(subject)

print("="*60)
print(subject)
print("="*60)

print("Shape :", imu_df.shape)

display(imu_df.head())

Subject36
Shape : (107760, 33)


,timestamp,time,q_w_chest,q_x_chest,q_y_chest,q_z_chest,q_w_left_hand,q_x_left_hand,q_y_left_hand,q_z_left_hand,...,g_x_left_knee,g_y_left_knee,g_z_left_knee,a_x_right_hand,a_y_right_hand,a_z_right_hand,g_x_right_hand,g_y_right_hand,g_z_right_hand,timestamp_unified
0,1.743521e+12,2025-04-01 11:22:22.002,0.847,-0.038,0.530,-0.017,0.901,-0.201,-0.379,-0.065,...,0.427,-0.671,-0.122,0.892,0.218,0.374,-1.280,0.000,-1.768,11:22:22 01/04/2025
1,1.743521e+12,2025-04-01 11:22:22.022,0.847,-0.038,0.531,-0.017,0.901,-0.200,-0.379,-0.067,...,0.488,0.061,1.220,0.891,0.216,0.371,-1.951,-1.098,-1.585,11:22:22 01/04/2025
2,1.743521e+12,2025-04-01 11:22:22.042,0.846,-0.038,0.531,-0.017,0.900,-0.201,-0.380,-0.068,...,0.671,0.549,2.073,0.900,0.217,0.378,-2.195,-2.195,-0.976,11:22:22 01/04/2025
3,1.743521e+12,2025-04-01 11:22:22.062,0.847,-0.038,0.531,-0.017,0.899,-0.201,-0.382,-0.069,...,0.793,1.037,2.073,0.900,0.216,0.377,-2.195,-2.317,-1.341,11:22:22 01/04/2025
4,1.743521e+12,2025-04-01 11:22:22.082,0.846,-0.038,0.531,-0.017,0.898,-0.201,-0.385,-0.069,...,0.061,1.768,1.890,0.892,0.210,0.374,-2.012,-0.549,-1.951,11:22:22 01/04/2025


In [39]:
imu_df.columns.tolist()

['timestamp',
 'time',
 'q_w_chest',
 'q_x_chest',
 'q_y_chest',
 'q_z_chest',
 'q_w_left_hand',
 'q_x_left_hand',
 'q_y_left_hand',
 'q_z_left_hand',
 'q_w_right_knee',
 'q_x_right_knee',
 'q_y_right_knee',
 'q_z_right_knee',
 'a_x_chest',
 'a_y_chest',
 'a_z_chest',
 'g_x_chest',
 'g_y_chest',
 'g_z_chest',
 'a_x_left_knee',
 'a_y_left_knee',
 'a_z_left_knee',
 'g_x_left_knee',
 'g_y_left_knee',
 'g_z_left_knee',
 'a_x_right_hand',
 'a_y_right_hand',
 'a_z_right_hand',
 'g_x_right_hand',
 'g_y_right_hand',
 'g_z_right_hand',
 'timestamp_unified']

In [41]:
# ==========================================================
# Cell 6 : Identify Sensor Columns
# ==========================================================

# Columns that should NOT be used for training
ignore_columns = [
    "timestamp",
    "time",
    "timestamp_unified"
]

# Sensor columns (30 features)
sensor_columns = [
    col for col in imu_df.columns
    if col not in ignore_columns
]

print("=" * 60)
print("Sensor Columns")
print("=" * 60)

print(f"Number of Sensor Features : {len(sensor_columns)}")

print("\nSensor Features:")

for col in sensor_columns:
    print(col)

Sensor Columns
Number of Sensor Features : 30

Sensor Features:
q_w_chest
q_x_chest
q_y_chest
q_z_chest
q_w_left_hand
q_x_left_hand
q_y_left_hand
q_z_left_hand
q_w_right_knee
q_x_right_knee
q_y_right_knee
q_z_right_knee
a_x_chest
a_y_chest
a_z_chest
g_x_chest
g_y_chest
g_z_chest
a_x_left_knee
a_y_left_knee
a_z_left_knee
g_x_left_knee
g_y_left_knee
g_z_left_knee
a_x_right_hand
a_y_right_hand
a_z_right_hand
g_x_right_hand
g_y_right_hand
g_z_right_hand


In [42]:
# ==========================================================
# Cell 7 : Fit StandardScaler
# ==========================================================

from sklearn.preprocessing import StandardScaler

print("Collecting training data...")

training_data = []

for subject in train_subjects:

    imu_path = os.path.join(
        DATASET_PATH,
        subject,
        f"IMU{subject}.csv"
    )

    imu_df = pd.read_csv(imu_path)

    training_data.append(
        imu_df[sensor_columns]
    )

training_df = pd.concat(training_data, ignore_index=True)

print("Training samples :", training_df.shape)

scaler = StandardScaler()

scaler.fit(training_df)

print("\n StandardScaler fitted successfully.")

Training samples : (5130454, 30)

✅ StandardScaler fitted successfully.


In [43]:
# ==========================================================
# Cell 8 : Process One Subject
# ==========================================================

def process_subject(subject):
    """
    Process one subject completely.

    Steps
    -----
    1. Load IMU data
    2. Keep only sensor columns
    3. Standardize using training scaler
    4. Return standardized IMU array
    """

    imu_path = os.path.join(
        DATASET_PATH,
        subject,
        f"IMU{subject}.csv"
    )

    imu_df = pd.read_csv(imu_path)

    # Keep only the 30 sensor features
    imu_features = imu_df[sensor_columns]

    # Standardize
    imu_scaled = scaler.transform(imu_features)

    return imu_scaled

In [59]:
# ==========================================================
# Cell 9 : Test Processing
# ==========================================================

subject = train_subjects[0]

#imu_scaled = process_subject(subject)
imu_scaled = process_subject(subject).astype(np.float32)

print("=" * 60)
print(subject)
print("=" * 60)

print("Scaled Shape :", imu_scaled.shape)

print("\nFirst 5 rows:")

print(imu_scaled[:5])

Subject36
Scaled Shape : (107760, 30)

First 5 rows:
[[ 1.3141339   0.3095813   0.48184294 -0.20595703  1.1450422  -0.03127709
   0.01383275 -0.13614789  1.4542694   0.2745861  -0.9517237  -0.35878205
   0.07370503 -0.11747231  1.0499159  -0.05500755 -0.31633756  0.03156624
   0.7992344  -0.62932265  0.80427873  0.03148036 -0.01274911  0.03188187
   0.66979647 -0.03090629  0.13384841 -0.02316522  0.00478004 -0.03627264]
 [ 1.3141339   0.3095813   0.48535028 -0.20595703  1.1450422  -0.027874
   0.01383275 -0.14083774  1.4542694   0.27110663 -0.9517237  -0.35878205
   0.06375839 -0.13077761  1.1539048  -0.00388944 -0.45800695  0.08014672
   0.82553166 -0.63517976  0.7958673   0.03266437 -0.002261    0.08486975
   0.66729796 -0.03462383  0.1261192  -0.03927273 -0.01339728 -0.03374727]
 [ 1.3098047   0.3095813   0.48535028 -0.20595703  1.1409984  -0.03127709
   0.01074685 -0.14318267  1.4542694   0.27110663 -0.9517237  -0.35878205
   0.09359832 -0.14408292  1.0904969   0.06188519 -0.455430

In [46]:
# ==========================================================
# Cell 10 : Create Sliding Windows
# ==========================================================

def create_sliding_windows(data,
                           window_size=128,
                           stride=10):
    """
    Convert IMU sequence into overlapping windows.

    Parameters
    ----------
    data : ndarray
        Shape = (num_samples, num_features)

    Returns
    -------
    ndarray
        Shape = (num_windows, window_size, num_features)
    """

    windows = []

    for start in range(
        0,
        len(data) - window_size + 1,
        stride
    ):

        end = start + window_size

        windows.append(
            data[start:end]
        )

    return np.asarray(windows)

In [47]:
# ==========================================================
# Cell 11 : Verify Sliding Windows
# ==========================================================

subject = train_subjects[0]

imu_scaled = process_subject(subject)

windows = create_sliding_windows(
    imu_scaled,
    WINDOW_SIZE,
    STRIDE
)

print("=" * 60)
print(subject)
print("=" * 60)

print("Original Shape :", imu_scaled.shape)
print("Window Shape   :", windows.shape)

print("\nOne Window Shape:")

print(windows[0].shape)

Subject36
Original Shape : (107760, 30)
Window Shape   : (10764, 128, 30)

One Window Shape:
(128, 30)


In [48]:
subject = train_subjects[0]

biomarker_path = os.path.join(
    DATASET_PATH,
    subject,
    f"Biomarkers{subject}.csv"
)

bio_df = pd.read_csv(biomarker_path)

print(bio_df.columns.tolist())

print("\nShape:", bio_df.shape)

display(bio_df.head())

['SpO2', 'HR', 'ActivityLabel']

Shape: (1070, 3)


,SpO2,HR,ActivityLabel
0,96.0,117.0,0.0
1,96.0,117.0,0.0
2,96.0,117.0,0.0
3,96.0,117.0,0.0
4,96.0,117.0,0.0


In [49]:
display(bio_df.tail())

,SpO2,HR,ActivityLabel
1065,94.0,132.0,0.0
1066,94.0,132.0,0.0
1067,94.0,132.0,0.0
1068,94.0,132.0,0.0
1069,94.0,132.0,0.0


In [50]:
print(bio_df["ActivityLabel"].value_counts().sort_index())

ActivityLabel
0.0    347
1.0    147
2.0    147
3.0    147
4.0    135
5.0    147
Name: count, dtype: int64


In [51]:
# ==========================================================
# Cell 12 : Expand Activity Labels to IMU Sampling Rate
# ==========================================================

def expand_activity_labels(activity_labels,
                           imu_length):
    """
    Expand activity labels so that they match
    the IMU sample length.

    Parameters
    ----------
    activity_labels : ndarray
        Original activity labels (1070)

    imu_length : int
        Number of IMU samples

    Returns
    -------
    ndarray
        Activity label for every IMU sample
    """

    repeat_factor = int(np.ceil(imu_length / len(activity_labels)))

    expanded = np.repeat(activity_labels,
                         repeat_factor)

    expanded = expanded[:imu_length]

    return expanded

In [52]:
# ==========================================================
# Cell 13 : Window Activity Labels
# ==========================================================

from scipy.stats import mode

def create_window_labels(expanded_labels,
                         window_size=128,
                         stride=10):
    """
    Create one activity label for each window
    using majority voting.
    """

    labels = []

    for start in range(
            0,
            len(expanded_labels)-window_size+1,
            stride):

        end = start + window_size

        window = expanded_labels[start:end]

        label = mode(
            window,
            keepdims=False
        ).mode

        labels.append(int(label))

    return np.array(labels)

In [53]:
# ==========================================================
# Cell 14 : Complete Subject Processing
# ==========================================================

def process_subject_complete(subject):
    """
    Complete preprocessing pipeline
    for one subject.
    """

    # ------------------------
    # IMU
    # ------------------------

    imu_scaled = process_subject(subject)

    windows = create_sliding_windows(
        imu_scaled,
        WINDOW_SIZE,
        STRIDE
    )

    # ------------------------
    # Biomarkers
    # ------------------------

    biomarker_path = os.path.join(
        DATASET_PATH,
        subject,
        f"Biomarkers{subject}.csv"
    )

    bio_df = pd.read_csv(biomarker_path)

    expanded_labels = expand_activity_labels(
        bio_df["ActivityLabel"].values,
        len(imu_scaled)
    )

    window_labels = create_window_labels(
        expanded_labels,
        WINDOW_SIZE,
        STRIDE
    )

    # ------------------------
    # Metadata
    # ------------------------

    row = metadata[
        metadata["Subject"] == subject
    ].iloc[0]

    gender = 0 if row["Gender"] == "M" else 1

    weight = row["Weight (kg)"]

    return (
        windows,
        window_labels,
        gender,
        weight
    )

In [55]:
# ==========================================================
# Cell 15 : Test Complete Pipeline
# ==========================================================

windows, activities, gender, weight = process_subject_complete(
    train_subjects[0]
)

print("="*60)

print("Windows Shape :", windows.shape)

print("Activity Labels :", activities.shape)

print("Gender :", gender)

print("Weight :", weight)

print("\nUnique Activities")

print(np.unique(activities))

Windows Shape : (10764, 128, 30)
Activity Labels : (10764,)
Gender : 0
Weight : 93.6

Unique Activities
[0 1 2 3 4 5]


In [56]:
# ==========================================================
# Cell 16 : Weight Classes
# ==========================================================

metadata["WeightClass"] = pd.qcut(
    metadata["Weight (kg)"],
    q=3,
    labels=[0,1,2]
)

print(metadata["WeightClass"].value_counts())

WeightClass
0    20
1    20
2    20
Name: count, dtype: int64


In [60]:
# ==========================================================
# Process and Save Training Subjects
# ==========================================================

OUTPUT_DIR = os.path.join(DATASET_PATH, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

for i, subject in enumerate(train_subjects):

    print(f"[{i+1}/{len(train_subjects)}] {subject}")

    windows, activity, gender, weight = process_subject_complete(subject)

    weight_class = metadata.loc[
        metadata["Subject"] == subject,
        "WeightClass"
    ].values[0]

    np.savez_compressed(
        os.path.join(OUTPUT_DIR, f"{subject}_train.npz"),

        windows=windows.astype(np.float32),

        activity=activity.astype(np.int64),

        gender=np.array([gender], dtype=np.int64),

        weight=np.array([weight_class], dtype=np.int64)
    )

    del windows
    del activity

    import gc
    gc.collect()

[1/48] Subject36
[2/48] Subject04
[3/48] Subject58
[4/48] Subject18
[5/48] Subject09
[6/48] Subject07
[7/48] Subject46
[8/48] Subject05
[9/48] Subject49
[10/48] Subject21
[11/48] Subject39
[12/48] Subject66
[13/48] Subject27
[14/48] Subject63
[15/48] Subject16
[16/48] Subject31
[17/48] Subject10
[18/48] Subject34
[19/48] Subject28
[20/48] Subject17
[21/48] Subject26
[22/48] Subject62
[23/48] Subject12
[24/48] Subject37
[25/48] Subject59
[26/48] Subject47
[27/48] Subject42
[28/48] Subject33
[29/48] Subject50
[30/48] Subject02
[31/48] Subject23
[32/48] Subject03
[33/48] Subject53
[34/48] Subject45
[35/48] Subject40
[36/48] Subject25
[37/48] Subject55
[38/48] Subject11
[39/48] Subject24
[40/48] Subject19
[41/48] Subject67
[42/48] Subject22
[43/48] Subject08
[44/48] Subject48
[45/48] Subject15
[46/48] Subject32
[47/48] Subject57
[48/48] Subject44


In [61]:
# ==========================================================
# Verify Processed Training Files
# ==========================================================

import os

files = sorted(os.listdir(OUTPUT_DIR))

print("Total Files:", len(files))

print("\nFirst 10 files:")

for f in files[:10]:
    print(f)

Total Files: 48

First 10 files:
Subject02_train.npz
Subject03_train.npz
Subject04_train.npz
Subject05_train.npz
Subject07_train.npz
Subject08_train.npz
Subject09_train.npz
Subject10_train.npz
Subject11_train.npz
Subject12_train.npz


In [62]:
sample_file = os.path.join(
    OUTPUT_DIR,
    files[0]
)

data = np.load(sample_file)

print(data.files)

print()

for key in data.files:
    print(key, data[key].shape, data[key].dtype)

['windows', 'activity', 'gender', 'weight']

windows (11044, 128, 30) float32
activity (11044,) int64
gender (1,) int64
weight (1,) int64


In [63]:
files = sorted(os.listdir(OUTPUT_DIR))

print(len(files))

48


In [64]:
# ==========================================================
# Process and Save Test Subjects
# ==========================================================

import gc
import os
import numpy as np

OUTPUT_DIR = os.path.join(DATASET_PATH, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

for i, subject in enumerate(test_subjects):

    print(f"[{i+1}/{len(test_subjects)}] {subject}")

    windows, activity, gender, weight = process_subject_complete(subject)

    weight_class = metadata.loc[
        metadata["Subject"] == subject,
        "WeightClass"
    ].values[0]

    np.savez_compressed(
        os.path.join(OUTPUT_DIR, f"{subject}_test.npz"),

        windows=windows.astype(np.float32),

        activity=activity.astype(np.int64),

        gender=np.array([gender], dtype=np.int64),

        weight=np.array([weight_class], dtype=np.int64)
    )

    del windows
    del activity

    gc.collect()

print("\n Test subjects processed successfully!")

[1/12] Subject01
[2/12] Subject06
[3/12] Subject41
[4/12] Subject51
[5/12] Subject14
[6/12] Subject60
[7/12] Subject38
[8/12] Subject54
[9/12] Subject13
[10/12] Subject64
[11/12] Subject52
[12/12] Subject56

 Test subjects processed successfully!


In [65]:
files = sorted(os.listdir(OUTPUT_DIR))

print("Total Files:", len(files))

train_files = [f for f in files if "_train.npz" in f]
test_files = [f for f in files if "_test.npz" in f]

print("Training Files:", len(train_files))
print("Test Files:", len(test_files))

Total Files: 60
Training Files: 48
Test Files: 12


In [66]:
sample = np.load(os.path.join(OUTPUT_DIR, "Subject02_train.npz"))

for key in sample.files:
    print(key, sample[key].shape, sample[key].dtype)

windows (11044, 128, 30) float32
activity (11044,) int64
gender (1,) int64
weight (1,) int64


In [67]:
print(sample["activity"][:10])
print(sample["gender"])
print(sample["weight"])

[0 0 0 0 0 0 0 0 0 0]
[0]
[1]
